In [ ]:
#use "../lib/nb.ml";;
load_css ()

# A Parse Table for a Shift-Reduce Parser

This notebook contains the parse table that is needed for a *shift reduce parser* that parses the following grammar:
$$
  \begin{eqnarray*}
  \mathrm{expr}        & \rightarrow & \mathrm{expr}\;\;\texttt{'+'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{expr}\;\;\texttt{'-'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{product}                                    \\[0.2cm]
  \mathrm{product}     & \rightarrow & \mathrm{product}\;\;\texttt{'*'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{product}\;\;\texttt{'/'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{factor}                                     \\[0.2cm]
  \mathrm{factor}      & \rightarrow & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}     \\
                       & \mid        & \texttt{NUMBER} 
  \end{eqnarray*}
$$

Below, we define the *grammar rules*.  The types `variable`, `token`, `symbol`, `state`, and `marked_rule` are all
strings; the type abbreviations only serve to document the meaning of the strings.  A rule is a pair consisting of its
variable and the list of the symbols of its body.

In [ ]:
type variable    = string
type token       = string
type symbol      = string
type state       = string
type marked_rule = string

In [ ]:
type rule = variable * symbol list

In [ ]:
let r1 : rule = ("e", [ "e"; "+"; "p" ])
let r2 : rule = ("e", [ "e"; "-"; "p" ])
let r3 : rule = ("e", [ "p" ])

let r4 : rule = ("p", [ "p"; "*"; "f" ])
let r5 : rule = ("p", [ "p"; "/"; "f" ])
let r6 : rule = ("p", [ "f" ])

let r7 : rule = ("f", [ "("; "e"; ")" ])
let r8 : rule = ("f", [ "NUMBER" ])

Next, we define the *action table* as a hash table, which is the mutable dictionary of OCaml.  In Python, an action is
either the string `'accept'`, a pair `('shift', state)`, or a pair `('reduce', rule)`.  In OCaml, the actions are the
constructors of the variant type `action`.  An entry that is missing in the table denotes the action `error`.

**Type safety:** In Python, a misspelled action like `('shfit', 's5')` is only detected when the parser stumbles
over it.  In OCaml, the compiler rejects it.

The *index operators* `.%{ }` and `.%{ } <-`, which are defined below, allow us to write the entries of the tables in the
same way as in Python: `actionTable.%{"s0", "("} <- Shift "s5"` corresponds to `actionTable['s0', '('] = ('shift', 's5')`.

In [ ]:
type action = Shift of state | Reduce of rule | Accept

let ( .%{} ) tbl key = Hashtbl.find tbl key
let ( .%{}<- ) tbl key value = Hashtbl.replace tbl key value

In [ ]:
let actionTable : (state * token, action) Hashtbl.t = Hashtbl.create 64
;;
actionTable.%{"s0", "("       } <- Shift "s5";
actionTable.%{"s0", "NUMBER"  } <- Shift "s2";

actionTable.%{"s1", "EOF"     } <- Reduce r6;
actionTable.%{"s1", "+"       } <- Reduce r6;
actionTable.%{"s1", "-"       } <- Reduce r6;
actionTable.%{"s1", "*"       } <- Reduce r6;
actionTable.%{"s1", "/"       } <- Reduce r6;
actionTable.%{"s1", ")"       } <- Reduce r6;

actionTable.%{"s2", "EOF"     } <- Reduce r8;
actionTable.%{"s2", "+"       } <- Reduce r8;
actionTable.%{"s2", "-"       } <- Reduce r8;
actionTable.%{"s2", "*"       } <- Reduce r8;
actionTable.%{"s2", "/"       } <- Reduce r8;
actionTable.%{"s2", ")"       } <- Reduce r8;

actionTable.%{"s3", "EOF"     } <- Reduce r3;
actionTable.%{"s3", "+"       } <- Reduce r3;
actionTable.%{"s3", "-"       } <- Reduce r3;
actionTable.%{"s3", "*"       } <- Shift "s12";
actionTable.%{"s3", "/"       } <- Shift "s11";
actionTable.%{"s3", ")"       } <- Reduce r3;

actionTable.%{"s4", "EOF"     } <- Accept;
actionTable.%{"s4", "+"       } <- Shift "s8";
actionTable.%{"s4", "-"       } <- Shift "s9";

actionTable.%{"s5", "("       } <- Shift "s5";
actionTable.%{"s5", "NUMBER"  } <- Shift "s2";

actionTable.%{"s6", "+"       } <- Shift "s8";
actionTable.%{"s6", "-"       } <- Shift "s9";
actionTable.%{"s6", ")"       } <- Shift "s7";

actionTable.%{"s7", "EOF"     } <- Reduce r7;
actionTable.%{"s7", "+"       } <- Reduce r7;
actionTable.%{"s7", "-"       } <- Reduce r7;
actionTable.%{"s7", "*"       } <- Reduce r7;
actionTable.%{"s7", "/"       } <- Reduce r7;
actionTable.%{"s7", ")"       } <- Reduce r7;

actionTable.%{"s8", "("       } <- Shift "s5";
actionTable.%{"s8", "NUMBER"  } <- Shift "s2";

actionTable.%{"s9", "("       } <- Shift "s5";
actionTable.%{"s9", "NUMBER"  } <- Shift "s2";

actionTable.%{"s10", "EOF"     } <- Reduce r2;
actionTable.%{"s10", "+"       } <- Reduce r2;
actionTable.%{"s10", "-"       } <- Reduce r2;
actionTable.%{"s10", "*"       } <- Shift "s12";
actionTable.%{"s10", "/"       } <- Shift "s11";
actionTable.%{"s10", ")"       } <- Reduce r2;

actionTable.%{"s11", "("       } <- Shift "s5";
actionTable.%{"s11", "NUMBER"  } <- Shift "s2";

actionTable.%{"s12", "("       } <- Shift "s5";
actionTable.%{"s12", "NUMBER"  } <- Shift "s2";

actionTable.%{"s13", "EOF"     } <- Reduce r4;
actionTable.%{"s13", "+"       } <- Reduce r4;
actionTable.%{"s13", "-"       } <- Reduce r4;
actionTable.%{"s13", "*"       } <- Reduce r4;
actionTable.%{"s13", "/"       } <- Reduce r4;
actionTable.%{"s13", ")"       } <- Reduce r4;

actionTable.%{"s14", "EOF"     } <- Reduce r5;
actionTable.%{"s14", "+"       } <- Reduce r5;
actionTable.%{"s14", "-"       } <- Reduce r5;
actionTable.%{"s14", "*"       } <- Reduce r5;
actionTable.%{"s14", "/"       } <- Reduce r5;
actionTable.%{"s14", ")"       } <- Reduce r5;

actionTable.%{"s15", "EOF"     } <- Reduce r1;
actionTable.%{"s15", "+"       } <- Reduce r1;
actionTable.%{"s15", "-"       } <- Reduce r1;
actionTable.%{"s15", "*"       } <- Shift "s12";
actionTable.%{"s15", "/"       } <- Shift "s11";
actionTable.%{"s15", ")"       } <- Reduce r1

Below is the definition of the *goto table*.

In [ ]:
let gotoTable : (state * variable, state) Hashtbl.t = Hashtbl.create 32
;;
gotoTable.%{"s0", "e"} <- "s4";
gotoTable.%{"s0", "p"} <- "s3";
gotoTable.%{"s0", "f"} <- "s1";

gotoTable.%{"s5", "e"} <- "s6";
gotoTable.%{"s5", "p"} <- "s3";
gotoTable.%{"s5", "f"} <- "s1";

gotoTable.%{"s8", "p"} <- "s15";
gotoTable.%{"s8", "f"} <- "s1";

gotoTable.%{"s9", "p"} <- "s10";
gotoTable.%{"s9", "f"} <- "s1";

gotoTable.%{"s11", "f"} <- "s14";
gotoTable.%{"s12", "f"} <- "s13"

Finally, we define the *state table*.  This is table is only used for pretty printing.  This table gives us a clue
about the information that is stored in the different stats.  The marked rules contain double quotes, so they are
written as *quoted strings* `{|...|}`.

In [ ]:
let stateTable : (state, marked_rule FSet.t) Hashtbl.t = Hashtbl.create 16
;;
stateTable.%{"s0"} <- FSet.of_list [ {|s -> • e|}; 
                      {|e -> • e "+" p|}; {|e -> • e "-" p|}; {|e -> • p|}; 
                      {|p -> • p "*" f|}; {|p -> • p "/" f|}; {|p -> • f|}; 
                      {|f -> • "(" e ")"|}; {|f -> • NUMBER|}
                    ];
stateTable.%{"s1"} <- FSet.of_list [ {|p -> f •|} ]; 
stateTable.%{"s2"} <- FSet.of_list [ {|f -> NUMBER •|} ]; 
stateTable.%{"s3"} <- FSet.of_list [ {|p -> p • "*" f|}; {|p -> p • "/" f|}; {|e -> p •|} ]; 
stateTable.%{"s4"} <- FSet.of_list [ {|S -> e •|}; {|e -> e • "+" p|}; {|e -> e • "-" p|} ];
stateTable.%{"s5"} <- FSet.of_list [ {|f -> "(" • e ")"|}; 
                      {|e -> • e "+" p|}; {|e -> • e "-" p|}; {|e -> • p|}; 
                      {|p -> • p "*" f|}; {|p -> • p "/" f|}; {|p -> • f|}; 
                      {|f -> • "(" e ")"|}; {|f -> • NUMBER|}
                    ];
stateTable.%{"s6"} <- FSet.of_list [ {|f -> "(" e • ")"|}; {|e -> e • "+" p|}; {|e -> e • "-" p|} ];
stateTable.%{"s7"} <- FSet.of_list [ {|f -> "(" e ")" •|} ];
stateTable.%{"s8"} <- FSet.of_list [ {|e -> e "+" • p|};
                      {|p -> • p "*" f|}; {|p -> • p "/" f|}; {|p -> • f|}; 
                      {|f -> • "(" e ")"|}; {|f -> • NUMBER|}
                    ];
stateTable.%{"s9"} <- FSet.of_list [ {|e -> e "-" • p|};
                      {|p -> • p "*" f|}; {|p -> • p "/" f|}; {|p -> • f|}; 
                      {|f -> • "(" e ")"|}; {|f -> • NUMBER|}
                    ];
stateTable.%{"s10"} <- FSet.of_list [ {|e -> e "-" p •|}; {|p -> p • "*" f|}; {|p -> p • "/" f|} ];
stateTable.%{"s11"} <- FSet.of_list [ {|p -> p "/" • f|}; {|f -> • "(" e ")"|}; {|f -> • NUMBER|} ]; 
stateTable.%{"s12"} <- FSet.of_list [ {|p -> p "*" • f|}; {|f -> • "(" e ")"|}; {|f -> • NUMBER|} ]; 
stateTable.%{"s13"} <- FSet.of_list [ {|p -> p "*" f •|} ]; 
stateTable.%{"s14"} <- FSet.of_list [ {|p -> p "/" f •|} ]; 
stateTable.%{"s15"} <- FSet.of_list [ {|e -> e "+" p •|}; {|p -> p • "*" f|}; {|p -> p • "/" f|} ]